# Ejercicio 3

Para resolver este problema usando Redes de Bayes, necesitamos modelar las relaciones entre las variables y luego calcular la probabilidad deseada utilizando las reglas de probabilidad y teorema de Bayes.

### Discretizar las variables GRE y GPA

Las variables GRE y GPA se van a discretizar en dos categorías cada una:

- GRE:
  - GRE ≥ 500 → 1
  - GRE < 500 → 0
- GPA:
  - GPA ≥ 3 → 1
  - GPA < 3 → 0

In [2]:
import pandas as pd

# Cargar los datos
df = pd.read_csv('binary.csv')

# Discretizar GRE y GPA
df['gre'] = (df['gre'] >= 500).astype(int)
df['gpa'] = (df['gpa'] >= 3).astype(int)

### Item a) Naive

Calcular la probabilidad de que una persona que proviene de una escuela con rango 1
no haya sido admitida en la universidad. P(ADMIT=0 ∣ RANK=1)

In [6]:
# count in df where admit = 0 and rank = 1
# count in df where rank = 1
# P(admit = 0 | rank = 1) = count(admit = 0 and rank = 1) / count(rank = 1)

PF_admit_0_rank_1 = df[(df['admit'] == 0) & (df['rank'] == 1)].shape[0]
PF_rank_1 = df[df['rank'] == 1].shape[0]

print("P(admit = 0 | rank = 1):")
print(PF_admit_0_rank_1/PF_rank_1)


P(admit = 0 | rank = 1):
0.45901639344262296


### Item b) Naive

Calcular la probabilidad de que una persona que fue a una escuela de rango 2, tenga
GRE = 450 y GPA = 3.5 sea admitida en la universidad. P(ADMIT=1 ∣ RANK=2, GRE=0, GPA=1)

In [24]:
PF_admit_1_rank_2_gre_0_gpa_1 = df[(df['admit'] == 1) & (df['rank'] == 2) & (df['gre'] == 0) & (df['gpa'] == 1)].shape[0]
PF_rank_2_gre_0_gpa_1 = df[(df['rank'] == 2) & (df['gre'] == 0) & (df['gpa'] == 1)].shape[0]

print("P(admit = 1 | rank = 2, gre = 0, gpa = 1):")
print(PF_admit_1_rank_2_gre_0_gpa_1/PF_rank_2_gre_0_gpa_1)

P(admit = 1 | rank = 2, gre = 0, gpa = 1):
0.19047619047619047


## Volviendo al ejercicio...

### Calcular las Probabilidades

Calculamos todas las probabilidades que pudieramos necesitar.

In [25]:
P_rank = df['rank'].value_counts(normalize=True).sort_index()
print("P(RANK = r):")
print(P_rank)

P(RANK = r):
rank
1    0.1525
2    0.3775
3    0.3025
4    0.1675
Name: proportion, dtype: float64


In [26]:
P_GRE_given_rank = df.groupby('rank')['gre'].value_counts(normalize=True).unstack().fillna(0)
print("P(GRE = e | RANK = r):")
print(P_GRE_given_rank)

P(GRE = e | RANK = r):
gre          0         1
rank                    
1     0.180328  0.819672
2     0.185430  0.814570
3     0.206612  0.793388
4     0.208955  0.791045


In [27]:
P_GPA_given_rank = df.groupby('rank')['gpa'].value_counts(normalize=True).unstack().fillna(0)
print("P(GPA = a | RANK = r):")
print(P_GPA_given_rank)

P(GPA = a | RANK = r):
gpa          0         1
rank                    
1     0.131148  0.868852
2     0.172185  0.827815
3     0.165289  0.834711
4     0.194030  0.805970


In [28]:
P_admit_given_conditions = df.groupby(['gpa', 'gre', 'rank'])['admit'].value_counts(normalize=True).unstack().fillna(0)
print("P(ADMIT = x | GPA = a, GRE = e, RANK = r):")
print(P_admit_given_conditions)

P(ADMIT = x | GPA = a, GRE = e, RANK = r):
admit                0         1
gpa gre rank                    
0   0   1     0.800000  0.200000
        2     0.571429  0.428571
        3     1.000000  0.000000
        4     1.000000  0.000000
    1   1     0.000000  1.000000
        2     0.842105  0.157895
        3     0.636364  0.363636
        4     0.888889  0.111111
1   0   1     0.500000  0.500000
        2     0.809524  0.190476
        3     0.812500  0.187500
        4     0.800000  0.200000
    1   1     0.446809  0.553191
        2     0.576923  0.423077
        3     0.752941  0.247059
        4     0.795455  0.204545


Sabiendo la dependencia de admit en GRE y GPA, podemos escribir la probabilidad a priori de admit como:

In [29]:
P_admit_given_conditions = df.groupby(['gpa', 'gre'])['admit'].value_counts(normalize=True).unstack().fillna(0)
print("P(ADMIT = x | GPA = a, GRE = e):")
print(P_admit_given_conditions)

P(ADMIT = x | GPA = a, GRE = e):
admit           0         1
gpa gre                    
0   0    0.840000  0.160000
    1    0.738095  0.261905
1   0    0.773585  0.226415
    1    0.642857  0.357143
